# Qwen3 0.6B Local Test

Use this notebook in VS Code with the repo `.venv` selected as the kernel.

This notebook is separate from the scratch tiny-transformer notebook. It loads the pretrained `Qwen/Qwen3-0.6B` model with Hugging Face `transformers`.

## 1. Find The Repo Root And Use The Current Kernel

In [ ]:
from pathlib import Path
import os
import shlex
import subprocess
import sys

cwd = Path.cwd().resolve()
project_dir = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / "pyproject.toml").exists() and (candidate / "step24_generate_qwen3.py").exists():
        project_dir = candidate
        break

if project_dir is None:
    raise RuntimeError("Could not find the repo root. Start VS Code from this project folder.")

os.chdir(project_dir)
print(f"project_dir: {project_dir}")
print(f"kernel python: {sys.executable}")

def run_py(script: str, *args: str) -> None:
    cmd = [sys.executable, "-u", script, *args]
    print(">", shlex.join(cmd))
    env = dict(os.environ)
    env["PYTHONUNBUFFERED"] = "1"
    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
        env=env,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, cmd)


## 2. Check CUDA

In [ ]:
import torch

print(f"torch: {torch.__version__}")
print(f"cuda available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"cuda device: {torch.cuda.get_device_name(0)}")


## 3. Download Qwen3 0.6B

Skip this cell if `models/qwen3-0.6b` already exists.

In [ ]:
run_py("download/step23_download_qwen3.py")


## 4. Ask A General Question

In [ ]:
run_py(
    "step24_generate_qwen3.py",
    "--device",
    "auto",
    "--prompt",
    "Where is Hong Kong?",
    "--max-new-tokens",
    "80",
    "--temperature",
    "0.3",
)


## 5. Test Grammar Correction

In [ ]:
run_py(
    "step24_generate_qwen3.py",
    "--device",
    "auto",
    "--prompt",
    "Correct this sentence: She go to school yesterday.",
    "--max-new-tokens",
    "80",
    "--temperature",
    "0.2",
)


## 6. Batch Test With Qwen Tokenizer And Weights

In [ ]:
run_py(
    "step27_test_qwen3.py",
    "--device",
    "auto",
    "--max-new-tokens",
    "96",
    "--temperature",
    "0.3",
)


## 7. Your Prompt

In [ ]:
prompt = "Explain why tiny models trained from scratch often give bad answers."

run_py(
    "step24_generate_qwen3.py",
    "--device",
    "auto",
    "--prompt",
    prompt,
    "--max-new-tokens",
    "160",
    "--temperature",
    "0.4",
)
